In [ ]:
import ee
import math
import geemap
import warnings
warnings.filterwarnings('ignore', message='Task was destroyed but it is pending')

# ── Authenticate & initialise the Earth Engine API ───────────────────────────
# This must be called before any ee.* objects are constructed.
# On first run it opens a browser auth flow; subsequent runs use cached creds.

ee.Authenticate()
ee.Initialize()

In [ ]:
# A blank map template should appear. If not, then the packages were not correctly instatlled

Map = geemap.Map()
Map

In [ ]:
# Input the AOI, and run this cell to view if your AOI is encompassed by the 1m DEM datset

# Load AOI
ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'

# Load AOI as Feature Collection
AOI = ee.FeatureCollection(ASSET_ID)

# Load dataset
dataset = ee.ImageCollection('USGS/3DEP/1m')

# Visualization parameters
visualization = {
    'min': 0,
    'max': 3000,
    'palette': [
        '3ae237', 'b5e22e', 'd6e21f', 'fff705', 'ffd611', 'ffb613', 'ff8b13',
        'ff6e08', 'ff500d', 'ff0000', 'de0101', 'c21301', '0602ff', '235cb1',
        '307ef3', '269db1', '30c8e2', '32d3ef', '3be285', '3ff38f', '86e26f'
    ]
}

# Center map
Map.setCenter(-98.0, 40, 4)

# Add layers
Map.addLayer(dataset, visualization, '1m DEM Dataset')

# Add your table (FeatureCollection) - make sure 'ASSET_ID' is defined
Map.addLayer(AOI, {'color': 'black'}, 'ASSET_ID')

# Display map
Map

In [ ]:
# =============================================================================
# 0.6m CHM EXTRACTION (NAIP)
# =============================================================================
#
# Dataset reference: https://github.com/smorf-ntsg/naip-chm
#
# ─────────────────────────────────────────────────────────────────────────────
# WORKFLOW JUSTIFICATION & OPERATION ORDER
# ─────────────────────────────────────────────────────────────────────────────
#
# Google Earth Engine resolves image projections LAZILY — it defers the
# actual pixel-grid computation until the last possible moment (export or
# getInfo). This means projection metadata attached to an image object is
# NOT enforced through subsequent operations unless you explicitly re-assert
# it at each step that can reset it.
#
# Three GEE operations are known projection-reset hazards:
#   1. mosaic()  — assembles tiles without enforcing a common grid
#   2. clip()    — strips or resets CRS context on the output image
#   3. band math (.divide(), .rename(), etc.) — inherits whatever projection
#                  the input image claims, which may be ambiguous post-mosaic
#
# The operation order for a defensible CHM pipeline is:
#
#   STEP 1 — LOAD tiles from the CHM collection filtered to AOI and year
#   STEP 2 — RESAMPLE + REPROJECT each tile to the analysis grid BEFORE mosaicking
#             (bilinear resample set before reproject so it governs interpolation;
#              per-tile reproject guarantees every tile is on the same
#              pixel lattice before they are combined)
#   STEP 3 — MOSAIC the reprojected tiles into a continuous surface
#             (all tiles are now on the same grid, so mosaic is deterministic)
#   STEP 4 — ASSERT analysis grid on mosaic result
#             (re-stamps projection on the merged image
#              in case mosaic() resets CRS context on the output)
#   STEP 5 — APPLY SCALE FACTOR (integer → meters conversion)
#             (non-spatial band math; no reproject needed here)
#   STEP 6 — CLIP to AOI boundary
#             (clip() is a known projection-reset hazard in GEE)
#   STEP 7 — ASSERT analysis grid on clipped output
#             (re-lock after clip; this is the final spatial op)
#   STEP 8 — CAST to float + ASSIGN NoData LAST
#             (NoData assignment must come AFTER all spatial operations;
#              applying it before reproject pulls -9999 fill values into
#              resampling kernels at AOI edges, corrupting border cells)
#   STEP 9 — EXPORT with explicit CRS + scale
#             (redundant enforcement at export level closes any remaining
#              projection ambiguity in the task submission)
#
# This order ensures projection context is explicit and stable at every
# operation that can alter pixel geometry. Each reproject() call is not
# redundant — each addresses a specific known GEE hazard.
#
# RESAMPLING METHOD — BILINEAR:
#   Bilinear resampling computes each output pixel as a weighted average of
#   the 4 nearest source pixels. It is the correct method for continuous
#   surfaces like canopy height because interpolating between known values
#   is physically meaningful.
#   DO NOT use bilinear for categorical rasters (land cover, soil type) —
#   use nearest neighbor for those, as averaging class codes is nonsensical.
#
# MOSAIC TILE PRIORITY:
#   The NAIP CHM is a preprocessed annual product — tiles within a given
#   year are harmonized and do not carry conflicting acquisition dates.
#   mosaic() is therefore a spatial merge only; no temporal priority is
#   imposed and no sort is needed.
#
# ─────────────────────────────────────────────────────────────────────────────



# =============================================================================
# USER CONFIGURATION
# =============================================================================

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection defining study corridor / analysis boundary.

Export_CRS = 'EPSG:5070'
# ^ Output coordinate reference system.

FOLDER = 'GEE_Exports_CHM'
# ^ Google Drive export folder for output rasters.
# ^ Folder is created automatically if it does not exist.

FILE_PREFIX = 'MN_I-90_CHM'
# ^ Output naming convention for reproducibility tracking.

EXPORT_SCALE = 0.6
# ^ Native NAIP CHM resolution (0.6 meters).
# ^ Resampling to a coarser resolution would introduce canopy generalization
#   and is not appropriate for corridor-scale vegetation structure analysis.

NODATA_VALUE = -9999
# ^ GIS-compatible NoData sentinel value.
# ^ -9999 is the conventional NoData for ArcGIS / QGIS raster interoperability.
# ^ Applied AS THE LAST OPERATION — see workflow justification above.

RESAMPLE_METHOD = 'bilinear'
# ^ Resampling interpolation method applied during per-tile reproject (Step 2).
# ^ Bilinear: weighted average of 4 nearest source pixels.
# ^ Appropriate for canopy height and all continuous surface derivatives.
# ^ .resample() must be called BEFORE .reproject() — it sets the interpolation
#   method that reproject will use. Reversed order has no effect.

CHM_YEAR = 2023
# ^ Selects CHM dataset year. 2023 is the most recent available.

CHM_SCALE_FACTOR = 100.0
# ^ Dataset stores values as integers ×100 → divide to convert back to meters.
# ^ Leave unchanged (dataset-defined scaling convention).

# =============================================================================
# DEFINE ANALYSIS GRID
# =============================================================================

analysis_grid = ee.Projection(Export_CRS).atScale(EXPORT_SCALE)
# ^ Defines the raster lattice for this entire pipeline.
# ^ This object is the single source of truth for all reproject() calls.
# ^ Defined early so it is available to every downstream operation.
# ^ Every reproject() call in this script references this object —
#   if the CRS or scale ever needs to change, it changes HERE only.

# =============================================================================
# LOAD AOI
# =============================================================================

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygon(s) from Earth Engine asset.

export_poly = export_poly.filter(ee.Filter.notNull(['system:index']))
# ^ Defensive filter: removes features with null system:index.
# ^ Prevents geometry operation failures on corrupted/incomplete features.
# ^ Standard geospatial step for any production pipeline.

AOI = export_poly.geometry()
# ^ Extracts geometry from the FeatureCollection.
# ^ AOI should be dissolved in ArcGIS Pro prior to ingestion for
#   deterministic analysis extent.
# ^ Used as the spatial reference for all filterBounds, clip, and export calls.

# =============================================================================
# CHM COLLECTION LOAD
# =============================================================================

chm_collection = (
    ee.ImageCollection('projects/naip-chm/assets/conus-structure-model')
    .filter(ee.Filter.eq('year', CHM_YEAR))
    # ^ Filters to the specified annual CHM product.
    # ^ NAIP CHM tiles within a given year are preprocessed into a
    #   harmonized annual product — no temporal sorting is required.

    .filterBounds(AOI)
    # ^ Restricts collection to tiles with spatial overlap with AOI.
    # ^ Avoids loading tiles with no analytical value.
)

# =============================================================================
# TILE PROVENANCE
# =============================================================================

tile_ids = chm_collection.aggregate_array('system:index').getInfo()

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("CHM COLLECTION LOG")
print("Source: NAIP CHM (CONUS Structure Model)")
print("Year:", CHM_YEAR)
print("Resample Method:", RESAMPLE_METHOD)
print("Mosaic Priority: Spatial merge only (harmonized annual product)")
print("")
print("TILES USED (pre-mosaic):")

for t in tile_ids:
    print(" -", t)

print("")
print("Total Tiles Intersecting AOI:", len(tile_ids))
print("Collection Size (server-side):", chm_collection.size().getInfo())
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# =============================================================================
# STEP 2 — PER-TILE RESAMPLE + REPROJECT (CRITICAL: BEFORE MOSAIC)
# =============================================================================

chm_collection_projected = chm_collection.map(
    lambda img: img.resample(RESAMPLE_METHOD).reproject(analysis_grid)
)
# ^ MOST CRITICAL OPERATION IN THE PIPELINE.
# ^
# ^ Problem this solves:
#   NAIP CHM tiles may be distributed across different native projections
#   depending on NAIP acquisition zone. Tiles from different zones have:
#     - Different coordinate origins
#     - Different pixel grid alignments
#   mosaic() does NOT reproject tiles before combining them. If tiles arrive
#   in mixed projections, the mosaic pixel grid is undefined/ambiguous.
# ^
# ^ Why per-tile (not post-mosaic):
#   Reprojecting AFTER mosaic() is too late. mosaic() has already resolved
#   the pixel geometry using whatever CRS GEE chose as the output default.
#   Per-tile reproject ensures every tile is snapped to the SAME pixel lattice
#   BEFORE they are combined. The mosaic then operates on geometrically
#   consistent inputs.
# ^
# ^ Why .resample() before .reproject():
#   .resample() sets the interpolation method that .reproject() will use
#   when transforming pixel values onto the new grid. Order is mandatory —
#   calling .resample() after .reproject() has no effect on the already-
#   completed transformation.

# =============================================================================
# STEP 3 — MOSAIC
# =============================================================================

chm_mosaic_raw = chm_collection_projected.mosaic()
# ^ Combines reprojected tiles into a continuous canopy height surface.
# ^ Because all tiles were reprojected to analysis_grid BEFORE this call,
#   the mosaic operates on a consistent pixel lattice.
# ^ No temporal priority needed — tiles are from a harmonized annual product.

# =============================================================================
# STEP 4 — ASSERT ANALYSIS GRID ON MOSAIC
# =============================================================================

chm_mosaic_locked = chm_mosaic_raw.reproject(analysis_grid)
# ^ Re-stamps the analysis grid on the mosaic output.
# ^ mosaic() CAN reset or leave ambiguous the projection metadata on its output
#   image even when all inputs share the same CRS. This call eliminates that
#   ambiguity by explicitly re-asserting the grid on the merged image.
# ^ This is not redundant with the per-tile reproject above — it addresses
#   a separate, downstream hazard introduced by the mosaic() operation itself.

# =============================================================================
# CHM DIAGNOSTIC: RAW vs LOCKED PROJECTION (QA CHECKPOINT)
# =============================================================================

print("CHM RAW MOSAIC PROJECTION (pre-lock — diagnostic only):")
print(chm_mosaic_raw.projection().getInfo())
# ^ Documents what projection GEE assigned to the raw mosaic output.
# ^ Expected: may be ambiguous or fallback CRS — this is the hazard
#   being addressed by Step 4.

print("CHM LOCKED PROJECTION (post-reproject — operational value):")
print(chm_mosaic_locked.projection().getInfo())
# ^ Should match analysis_grid (Export_CRS @ EXPORT_SCALE).

# =============================================================================
# STEP 5 — APPLY SCALE FACTOR (INTEGER → METERS)
# =============================================================================

chm_mosaic = (
    chm_mosaic_locked
    .divide(CHM_SCALE_FACTOR)
    # ^ Converts dataset's scaled integer storage values to meters.
    # ^ Dataset convention: stored value = canopy height (m) × 100.
    # ^ Non-spatial operation — does not affect pixel geometry or projection.

    .rename('canopy_height_m')
    # ^ Names output band explicitly for downstream GIS identification.
)

# =============================================================================
# CHM QA STATISTICS (VALIDATION CHECKPOINT)
# =============================================================================

chm_stats = chm_mosaic.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=AOI,
    scale=EXPORT_SCALE,
    bestEffort=False,
    # ^ Fails explicitly if pixel count exceeds maxPixels.
    # ^ bestEffort=True silently coarsens the scale to fit pixel limit —
    #   this is acceptable for visualization but not for QA validation.
    maxPixels=1e13
)

print("CHM STATS (min/max canopy height, meters):", chm_stats.getInfo())
# ^ Expected range: 0m (bare ground) to ~40m (tall canopy) for MN corridor.
# ^ Values outside this range indicate scale factor misapplication or
#   mosaic construction failure.

# =============================================================================
# STEP 6 — CLIP TO AOI
# =============================================================================

chm_clipped = chm_mosaic.clip(AOI)
# ^ Constrains output raster to AOI boundary polygon.
# ^ WARNING: clip() is a known projection-reset operation in GEE.
#   It strips or leaves ambiguous the projection metadata on its output.
#   DO NOT treat chm_clipped as having a guaranteed projection —
#   Step 7 re-asserts the grid immediately after this call.

# =============================================================================
# STEP 7 — ASSERT ANALYSIS GRID AFTER CLIP (FINAL SPATIAL LOCK)
# =============================================================================

chm_final = chm_clipped.reproject(analysis_grid)
# ^ Re-asserts the analysis grid after clip().
# ^ This is the FINAL spatial operation on the raster.
# ^ After this call, chm_final has an explicit, stable projection that
#   will govern the export pixel grid.
# ^ The subsequent toFloat() and NoData assignment calls are non-spatial
#   (they modify pixel values, not pixel geometry) and do not require reproject().

# =============================================================================
# STEP 8 — CAST + ASSIGN NoData (NON-SPATIAL OPERATIONS, APPLIED LAST)
# =============================================================================

ready_to_export = (
    chm_final
    .toFloat()
    # ^ Casts to 32-bit float.
    # ^ Preserves sub-meter precision in the continuous canopy height surface.

    .where(chm_final.mask().Not(), NODATA_VALUE)
    # ^ Assigns -9999 to all masked pixels (outside AOI, missing CHM data).
    # ^ MUST BE THE LAST OPERATION.
    # ^ If NoData is assigned before reproject() or clip(), the -9999 fill
    #   value participates in bilinear resampling kernels at the AOI boundary,
    #   pulling fill values into valid edge cells and corrupting border pixels.
)

# =============================================================================
# PIXEL COUNT QA (SANITY CHECK)
# =============================================================================

print("CHM pixel count (QA check):",
    chm_final.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=AOI,
        scale=EXPORT_SCALE,
        bestEffort=False,
        # ^ Explicit failure on pixel limit breach — no silent scale degradation.
        maxPixels=1e13
    ).getInfo()
)
# ^ Confirms the CHM surface covers the expected AOI extent.
# ^ A count of 0 or unexpectedly low count indicates a clip, reproject, or
#   mosaic failure and should halt the pipeline before export.

# =============================================================================
# VISUALIZATION — SINGLE MAP (CHM + AOI)
# =============================================================================
# ^ Single map instance with both products as toggleable layers.
# ^ Layer order: CHM (bottom) → AOI boundary (top).

Map = geemap.Map()
Map.centerObject(AOI, 11)
Map.add_basemap('SATELLITE')

Map.addLayer(
    chm_final,
    # ^ Visualize chm_final (post-clip, post-reproject) — the actual
    #   export product, not an intermediate derivative.
    {
        'min': 0,
        'max': 40,
        # ^ Canopy height range in meters.
        # ^ 0 = bare ground; 40m = tall canopy for MN I-90 corridor.
        'palette': [
            'ffffff',  # 0m: bare ground
            'c7e9b4',  # low canopy
            '7fcdbb',  # low-mid canopy
            '41b6c4',  # mid canopy
            '2c7fb8',  # tall canopy
            '253494'   # maximum canopy height
        ]
    },
    'CHM (m) — EXPORT PRODUCT'
)

Map.addLayer(AOI, {'color': 'FF1744'}, 'AOI Boundary')

display(Map)

# =============================================================================
# STEP 9 — EXPORT (EXPLICIT CRS + SCALE ENFORCEMENT AT TASK LEVEL)
# =============================================================================

task = ee.batch.Export.image.toDrive(
    image=ready_to_export,
    # ^ Final CHM raster: canopy height in meters, float32, -9999 NoData.

    description=FILE_PREFIX,
    # ^ Task label in Earth Engine Task Manager.

    folder=FOLDER,
    # ^ Output Google Drive folder.

    fileNamePrefix=FILE_PREFIX,
    # ^ Output filename base.

    region=AOI,
    # ^ Export boundary (matches analysis AOI exactly).

    scale=EXPORT_SCALE,
    # ^ Pixel resolution (0.6m native CHM resolution).
    # ^ Explicit here even though image is already reprojected —
    #   redundant enforcement closes any remaining ambiguity in the
    #   task submission layer.

    crs=Export_CRS,
    # ^ Output CRS.
    # ^ Explicit here even though image is already reprojected —
    #   same rationale: enforce at every layer.

    maxPixels=int(1e13),
    # ^ Prevents export failure on large corridor datasets.

    formatOptions={
        'noData': NODATA_VALUE
    }
    # ^ Registers NoData sentinel in the output GeoTIFF metadata.
    # ^ Ensures ArcGIS / QGIS correctly interprets masked pixels.
)

task.start()

# =============================================================================
# FINAL STATUS BLOCK
# =============================================================================

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("CHM EXPORT PIPELINE INITIATED")
print("")
print("OPERATION ORDER:")
print("  1.  Load tiles (year filter + AOI bounds)")
print("  2.  Resample (bilinear) + reproject each tile (per-tile, PRE-mosaic)")
print("  3.  Mosaic reprojected tiles (spatial merge — harmonized annual product)")
print("  4.  Assert analysis grid on mosaic output")
print("  5.  Apply scale factor (integer → meters)")
print("  6.  Clip CHM to AOI")
print("  7.  Assert analysis grid post-clip (final spatial lock)")
print("  8.  Cast to float32 + assign NoData (non-spatial, applied last)")
print("  9.  Export (explicit CRS + scale at task level)")
print("")
print("Grid Standard:    ", Export_CRS, "@", EXPORT_SCALE, "m")
print("Resample Method:  ", RESAMPLE_METHOD)
print("CHM Year:         ", CHM_YEAR)
print("Scale Factor:     ", CHM_SCALE_FACTOR, "(integer → meters)")
print("Output:            Canopy height (meters), float32, NoData =", NODATA_VALUE)
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

In [ ]:
# =============================================================================
# 1m DEM EXPORT + SLOPE EXTRACTION (PERCENT)   
# =============================================================================
#
# ─────────────────────────────────────────────────────────────────────────────
# WORKFLOW JUSTIFICATION & OPERATION ORDER
# ─────────────────────────────────────────────────────────────────────────────
#
# Google Earth Engine resolves image projections LAZILY — it defers the
# actual pixel-grid computation until the last possible moment (export or
# getInfo). This means projection metadata attached to an image object is
# NOT enforced through subsequent operations unless you explicitly re-assert
# it at each step that can reset it.
#
# Three GEE operations are known projection-reset hazards:
#   1. mosaic()  — assembles tiles without enforcing a common grid
#   2. clip()    — strips or resets CRS context on the output image
#   3. terrain derivatives (slope/aspect) — inherit whatever projection
#                  the input image claims, which may be ambiguous post-mosaic
#
# The operation order for a defensible slope pipeline is:
#
#   STEP 1 — LOAD individual tiles from the collection
#   STEP 2 — RESAMPLE + REPROJECT each tile to the analysis grid BEFORE mosaicking
#             (bilinear resample set before reproject so it governs interpolation;
#              per-tile reproject guarantees every tile is on the same
#              pixel lattice before they are combined)
#   STEP 3 — MOSAIC the reprojected tiles into a continuous surface
#             (all tiles are now on the same grid, so mosaic is deterministic)
#   STEP 4 — ASSERT analysis grid on mosaic result
#             (re-stamps projection on the merged image
#              in case mosaic() resets CRS context on the output)
#   STEP 5 — EXPORT DEM (same grid standard as slope — parallel output product)
#   STEP 6 — COMPUTE SLOPE on the locked-grid DEM
#             (3x3 neighborhood now operates on known, consistent pixel geometry)
#   STEP 7 — ASSERT analysis grid on slope output
#             (slope() can reset projection; re-lock before any further ops)
#   STEP 8 — CONVERT DEGREES TO PERCENT SLOPE
#   STEP 9 — CLIP to AOI boundary
#             (clip() is a known projection-reset hazard in GEE)
#   STEP 10 — ASSERT analysis grid on clipped output
#             (re-lock after clip; this is the final spatial op)
#   STEP 11 — CAST to float + UNMASK nodata LAST
#             (unmask must come AFTER all spatial operations; applying it
#              before reproject pulls -9999 fill values into resampling
#              at AOI edges, corrupting border cells)
#   STEP 12 — EXPORT SLOPE with explicit CRS + scale
#             (redundant enforcement at export level closes any remaining
#              projection ambiguity in the task submission)
#
# This order ensures projection context is explicit and stable at every
# operation that can alter pixel geometry. Each reproject() call is not
# redundant — each addresses a specific known GEE hazard.
#
# RESAMPLING METHOD — BILINEAR:
#   Bilinear resampling computes each output pixel as a weighted average of
#   the 4 nearest source pixels. It is the correct method for continuous
#   terrain surfaces (elevation, slope) because interpolating between known
#   values is physically meaningful for a smooth surface.
#   Standardizing to bilinear here means this script is directly portable
#   to a 10m workflow without modification — at 10m (where source data is
#   typically in geographic/degree-unit CRS), nearest neighbor introduces
#   5–9m pixel assignment error that bilinear eliminates.
#   DO NOT use bilinear for categorical rasters (land cover, soil type) —
#   use nearest neighbor for those, as averaging class codes is nonsensical.
#
# MOSAIC TILE PRIORITY:
#   mosaic() uses last-on-top ordering by collection sort order.
#   Sorting by system:time_start DESCENDING means the NEWEST tile wins
#   in overlapping areas. For USGS 3DEP, newer acquisitions reflect
#   updated survey campaigns and should be preferred over older data.
#
# ─────────────────────────────────────────────────────────────────────────────



# =============================================================================
# USER CONFIGURATION
# =============================================================================

ASSET_ID = 'projects/ee-kylesteen/assets/I90_ROW_Buf_1500ft'
# ^ AOI polygon FeatureCollection defining study corridor / analysis boundary.

Export_CRS = 'EPSG:5070'
# ^ Output coordinate reference system.

FOLDER = 'GEE_Exports_Slope'
# ^ Google Drive export folder for output rasters.
# ^ Folder is created automatically if it does not exist.

FILE_PREFIX_SLOPE = 'MN_I-90_SLOPE_PCT'
FILE_PREFIX_DEM   = 'MN_I-90_DEM'
# ^ Output naming conventions for reproducibility tracking.
# ^ Separate prefixes allow slope and DEM outputs to coexist in the same folder.

EXPORT_SCALE = 1
# ^ Output resolution in meters.
# ^ Downsampling to coarser resolution would introduce terrain generalization
#   and is not appropriate for corridor-scale slope analysis.
# ^ To run a 10m workflow: change this value to 10 and update the
#   ImageCollection ID to 'USGS/3DEP/10m_collection'. All other pipeline logic is
#   identical — bilinear resampling handles the CRS difference correctly.

NODATA_VALUE = -9999
# ^ GIS-compatible NoData sentinel value.
# ^ -9999 is the conventional NoData for ArcGIS / QGIS raster interoperability.
# ^ Applied via unmask() AS THE LAST OPERATION — see workflow justification above.

RESAMPLE_METHOD = 'bilinear'
# ^ Resampling interpolation method applied during per-tile reproject (Step 2).
# ^ Bilinear: weighted average of 4 nearest source pixels.
# ^ Appropriate for elevation and all continuous terrain derivatives.
# ^ Standardized here so 1m and 10m pipelines use identical methodology.
# ^ .resample() must be called BEFORE .reproject() — it sets the interpolation
#   method that reproject will use. Reversed order has no effect.

# =============================================================================
# DEFINE ANALYSIS GRID
# =============================================================================

analysis_grid = ee.Projection(Export_CRS).atScale(EXPORT_SCALE)
# ^ Defines the raster lattice for this entire pipeline.
# ^ This object is the single source of truth for all reproject() calls.
# ^ Defined early so it is available to every downstream operation.
# ^ Every reproject() call in this script references this object —
#   if the CRS or scale ever needs to change, it changes HERE only.

# =============================================================================
# LOAD AOI
# =============================================================================

export_poly = ee.FeatureCollection(ASSET_ID)
# ^ Loads AOI polygon(s) from Earth Engine asset.

export_poly = export_poly.filter(ee.Filter.notNull(['system:index']))
# ^ Defensive filter: removes features with null system:index.
# ^ Prevents geometry operation failures on corrupted/incomplete features.
# ^ Standard geospatial step for any production pipeline.

AOI = export_poly.geometry()
# ^ Extracts geometry from the FeatureCollection.
# ^ Used as the spatial reference for all filterBounds, clip, and export calls.

# =============================================================================
# DEM COLLECTION LOAD
# =============================================================================

dem_collection = (
    ee.ImageCollection('USGS/3DEP/1m')
    .filterBounds(AOI)
    # ^ Restricts collection to tiles with spatial overlap with AOI.
    # ^ Avoids loading tiles with no analytical value.

    .select('elevation')
    # ^ Selects the elevation band by NAME, not by index position.

    .sort('system:time_start', False)
    # ^ Sorts collection by acquisition date, DESCENDING (newest first).
    # ^ mosaic() composites using last-on-top ordering relative to collection
    #   sort order — the FIRST item in the sorted collection = bottom of stack.
    # ^ Descending sort = NEWEST tile wins in overlapping areas.
    # ^ USGS 3DEP newer acquisitions reflect updated survey campaigns and
    #   should be preferred over older data in overlap zones.
)

# =============================================================================
# TILE PROVENANCE
# =============================================================================

tile_ids = dem_collection.aggregate_array('system:index').getInfo()

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("DEM COLLECTION LOG")
print("Source: USGS 3DEP")
print("Band Selected: elevation")
print("Resample Method:", RESAMPLE_METHOD)
print("Mosaic Priority: DESCENDING by system:time_start (newest tile wins)")
print("")
print("TILES USED (pre-mosaic, in mosaic priority order):")

for t in tile_ids:
    print(" -", t)

print("")
print("Total Tiles Intersecting AOI:", len(tile_ids))
print("Collection Size (server-side):", dem_collection.size().getInfo())
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# =============================================================================
# STEP 2 — PER-TILE RESAMPLE + REPROJECT (CRITICAL: BEFORE MOSAIC)
# =============================================================================

dem_collection_projected = dem_collection.map(
    lambda img: img.resample(RESAMPLE_METHOD).reproject(analysis_grid)
)
# ^ MOST CRITICAL OPERATION IN THE PIPELINE.
# ^
# ^ Problem this solves:
#   USGS 3DEP 1m tiles are distributed in UTM zones.
#   Tiles from different zones have:
#     - Different coordinate origins
#     - Different pixel grid alignments
#     - Different proj4 parameters (false easting, central meridian)
#   mosaic() does NOT reproject tiles before combining them. If tiles arrive
#   in mixed projections, the mosaic pixel grid is undefined/ambiguous, and
#   the 3x3 neighborhood kernel used by ee.Terrain.slope() operates on
#   misaligned pixels — producing incorrect slope values at tile boundaries
#   and potentially across the entire study area.
# ^
# ^ Why per-tile (not post-mosaic):
#   Reprojecting AFTER mosaic() is too late. mosaic() has already resolved
#   the pixel geometry using whatever CRS GEE chose as the output default.
#   Per-tile reproject ensures every tile is snapped to the SAME pixel lattice
#   BEFORE they are combined. The mosaic then operates on geometrically
#   consistent inputs.
# ^
# ^ Why .resample() before .reproject():
#   .resample() sets the interpolation method that .reproject() will use
#   when transforming pixel values onto the new grid. Order is mandatory —
#   calling .resample() after .reproject() has no effect on the already-
#   completed transformation.

# =============================================================================
# STEP 3 — MOSAIC
# =============================================================================

dem_mosaic_raw = dem_collection_projected.mosaic()
# ^ Combines reprojected tiles into a continuous elevation surface.
# ^ Because all tiles were reprojected to analysis_grid BEFORE this call,
#   the mosaic operates on a consistent pixel lattice.
# ^ Overlap resolution: last-on-top = newest tile wins (per sort above).

# =============================================================================
# STEP 4 — ASSERT ANALYSIS GRID ON MOSAIC
# =============================================================================

dem_mosaic = dem_mosaic_raw.reproject(analysis_grid)
# ^ Re-stamps the analysis grid on the mosaic output.
# ^ mosaic() CAN reset or leave ambiguous the projection metadata on its output
#   image even when all inputs share the same CRS. This call eliminates that
#   ambiguity by explicitly re-asserting the grid on the merged image.
# ^ This is not redundant with the per-tile reproject above — it addresses
#   a separate, downstream hazard introduced by the mosaic() operation itself.

# =============================================================================
# DEM DIAGNOSTIC: RAW vs LOCKED PROJECTION (QA CHECKPOINT)
# =============================================================================

print("DEM RAW MOSAIC PROJECTION (pre-lock — diagnostic only):")
print(dem_mosaic_raw.projection().getInfo())
# ^ Documents what projection GEE assigned to the raw mosaic output.
# ^ Expected: may be ambiguous, fallback, or mixed — this is the hazard
#   being addressed by Step 4.

print("DEM LOCKED PROJECTION (post-reproject — operational value):")
print(dem_mosaic.projection().getInfo())
# ^ Should match analysis_grid (Export_CRS @ EXPORT_SCALE).
# ^ This is the projection that will govern slope kernel computation.

# =============================================================================
# DEM QA STATISTICS (VALIDATION CHECKPOINT)
# =============================================================================

dem_stats = dem_mosaic.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=AOI,
    scale=EXPORT_SCALE,
    bestEffort=False,
    # ^ Fails explicitly if pixel count exceeds maxPixels.
    # ^ bestEffort=True silently coarsens the scale to fit pixel limit —
    #   this is acceptable for visualization but not for QA validation.
    # ^ For a defensible methodology, we want explicit failure, not silent
    #   degradation of the validation check itself.
    maxPixels=1e13
)

print("DEM STATS (min/max elevation, meters):", dem_stats.getInfo())
# ^ Expected range for AOI. Values outside the plausible range indicate
#   mosaic construction failure or incorrect AOI geometry.

# =============================================================================
# STEP 5 — EXPORT DEM (PARALLEL OUTPUT PRODUCT)
# =============================================================================

dem_ready_to_export = (
    dem_mosaic
    .clip(AOI)
    # ^ Constrains DEM output to AOI boundary polygon.

    .reproject(analysis_grid)
    # ^ Re-asserts analysis grid after clip() — clip() is a known
    #   projection-reset hazard. Mirrors the same post-clip lock applied
    #   to the slope product for consistency.

    .toFloat()
    # ^ Casts to 32-bit float to preserve sub-meter elevation precision.

    .where(dem_mosaic.mask().Not(), NODATA_VALUE)
    # ^ Ensures correct NoData values
)

dem_task = ee.batch.Export.image.toDrive(
    image=dem_ready_to_export,
    # ^ Final DEM raster: elevation in meters, float32, -9999 NoData.
    # ^ Same grid standard as slope product.
    # ^ Exporting the DEM alongside slope allows downstream validation:
    #   slope values can be spot-checked against the source elevation surface.

    description=FILE_PREFIX_DEM,
    folder=FOLDER,
    fileNamePrefix=FILE_PREFIX_DEM,
    region=AOI,
    scale=EXPORT_SCALE,
    crs=Export_CRS,
    maxPixels=int(1e13),
    formatOptions={
        'noData': -9999}
    # ^ Ensures correct NoData values
)

dem_task.start()
print("DEM export task initiated:", FILE_PREFIX_DEM)

# =============================================================================
# STEP 6 — SLOPE DERIVATION (ON LOCKED ANALYSIS GRID)
# =============================================================================

slope_deg = ee.Terrain.slope(dem_mosaic)
# ^ Computes slope in degrees.
# ^ Because dem_mosaic is reprojected to analysis_grid (Step 4),
#   the kernel operates on a known, metrically consistent pixel geometry.
# ^ Output: angular slope in degrees (0–90°).
# ^ If dem_mosaic carried ambiguous projection metadata, this kernel would
#   operate on undefined pixel spacing, producing incorrect slope values.
#   The reproject() in Step 4 prevents this.

# =============================================================================
# STEP 7 — ASSERT ANALYSIS GRID ON SLOPE OUTPUT
# =============================================================================

slope_deg_locked = slope_deg.reproject(analysis_grid)
# ^ ee.Terrain.slope() is a known projection-reset operation in GEE.
# ^ It inherits the input image's projection but does not GUARANTEE the
#   output image carries the same explicit projection metadata.
# ^ This call re-locks the grid on the slope output before the percent
#   conversion math chain, ensuring subsequent operations remain on the
#   correct pixel lattice.

# =============================================================================
# STEP 8 — SLOPE UNIT CONVERSION: DEGREES → PERCENT
# =============================================================================

slope_pct = (
    slope_deg_locked
    .multiply(math.pi / 180)
    # ^ Degrees → radians.
    # ^ Required because GEE's .tan() operates in radians, not degrees.

    .tan()
    # ^ Radians → rise/run ratio.
    # ^ tan(θ) = vertical rise / horizontal run.
    # ^ This is the definition of percent slope before the ×100 scaling.
    # ^ Note: output domain is 0–∞ (not capped at 100%).
    #   A 45° slope = 100%; slopes steeper than 45° exceed 100%.
    #   This is correct behavior — percent slope is unbounded above 100%.

    .multiply(100)
    # ^ Rise/run ratio → percent slope (×100 scaling convention).

    .rename('slope_pct')
    # ^ Names output band explicitly for downstream GIS identification.
)

# =============================================================================
# STEP 9 — CLIP TO AOI
# =============================================================================

slope_clipped = slope_pct.clip(AOI)
# ^ Constrains output raster to AOI boundary polygon.
# ^ WARNING: clip() is a known projection-reset operation in GEE.
#   It strips or leaves ambiguous the projection metadata on its output.
#   DO NOT treat slope_clipped as having a guaranteed projection —
#   Step 10 re-asserts the grid immediately after this call.

# =============================================================================
# STEP 10 — ASSERT ANALYSIS GRID AFTER CLIP (FINAL SPATIAL LOCK)
# =============================================================================

slope_final = slope_clipped.reproject(analysis_grid)
# ^ Re-asserts the analysis grid after clip().
# ^ This is the FINAL spatial operation on the raster.
# ^ After this call, slope_final has an explicit, stable projection that
#   will govern the export pixel grid.
# ^ The subsequent toFloat() and unmask() calls are non-spatial (they
#   modify pixel values, not pixel geometry) and do not require reproject().

# =============================================================================
# STEP 11 — CAST + UNMASK (NON-SPATIAL OPERATIONS, APPLIED LAST)
# =============================================================================

slope_ready_to_export = (
    slope_final
    .toFloat()
    # ^ Casts to 32-bit float.
    # ^ Preserves sub-percent precision in the continuous slope surface.
    # ^ GEE default integer cast would truncate slope values to whole numbers,
    #   which is unacceptable for a 1m terrain derivative product.

    .where(slope_final.mask().Not(), NODATA_VALUE)
)

# =============================================================================
# PIXEL COUNT QA (SANITY CHECK)
# =============================================================================

print("Slope pixel count (QA check):",
    slope_final.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=AOI,
        scale=EXPORT_SCALE,
        bestEffort=False,
        # ^ Explicit failure on pixel limit breach — no silent scale degradation.
        maxPixels=1e13
    ).getInfo()
)
# ^ Confirms the slope surface covers the expected AOI extent.
# ^ A count of 0 or unexpectedly low count indicates a clip, reproject, or
#   mosaic failure and should halt the pipeline before export.

# =============================================================================
# VISUALIZATION — SINGLE MAP (DEM + SLOPE + AOI)
# =============================================================================
# ^ Single map instance with both products as toggleable layers.
# ^ Rendering both layers on one map object avoids the duplicate display()
#   issue where a second map re-renders all previously added layers.
# ^ Layer order: DEM (bottom) → Slope (top) → AOI boundary (always on top).

Map = geemap.Map()
Map.centerObject(AOI, 11)
Map.add_basemap('SATELLITE')

Map.addLayer(
    dem_mosaic.clip(AOI),
    # ^ DEM layer clipped to AOI for visualization only.
    # ^ Does not affect the analytical pipeline — dem_mosaic remains
    #   unclipped for slope computation (clip here produces a new image
    #   object; dem_mosaic is unchanged).
    {
        'min': 0,
        'max': 1,
        'palette': [
            '006633',  # low elevation
            'E5FFCC',  # low-mid
            'FFCC00',  # mid elevation
            '662A00',  # high elevation
            'FFFFFF'   # peaks / high relief
        ]
    },
    'DEM (QC - Ground Truth Check)'
)

Map.addLayer(
    slope_final,
    # ^ Visualize slope_final (post-clip, post-reproject) — the actual
    #   export product, not an intermediate derivative.
    {
        'min': 0,
        'max': 1,
        # ^ Percent slope visualization range.
        # ^ Values above 100% (>45° slopes) are valid but rare in MN I-90 corridor.
        'palette': [
            '006633',  # 0–20%: flat to gentle slope
            'E5FFCC',  # 20–40%: gentle to moderate slope
            'FFCC00',  # 40–60%: moderate to steep slope
            '662A00',  # 60–80%: steep slope
            'FFFFFF'   # 80–100%+: extreme slope
        ]
    },
    'SLOPE (%) — EXPORT PRODUCT'
)

Map.addLayer(AOI, {'color': 'FF1744'}, 'AOI Boundary')

display(Map)

# =============================================================================
# STEP 12 — EXPORT SLOPE (EXPLICIT CRS + SCALE ENFORCEMENT AT TASK LEVEL)
# =============================================================================

slope_task = ee.batch.Export.image.toDrive(
    image=slope_ready_to_export,
    # ^ Final slope raster: percent slope, float32, -9999 NoData.

    description=FILE_PREFIX_SLOPE,
    # ^ Task label in Earth Engine Task Manager.

    folder=FOLDER,
    # ^ Output Google Drive folder.

    fileNamePrefix=FILE_PREFIX_SLOPE,
    # ^ Output filename base.

    region=AOI,
    # ^ Export boundary (matches analysis AOI exactly).

    scale=EXPORT_SCALE,
    # ^ Pixel resolution.
    # ^ Explicit here even though image is already reprojected —
    #   redundant enforcement closes any remaining ambiguity in the
    #   task submission layer.

    crs=Export_CRS,
    # ^ Output CRS.
    # ^ Explicit here even though image is already reprojected —
    #   same rationale: enforce at every layer.

    maxPixels=int(1e13),
    # ^ Prevents export failure on large corridor datasets.

    formatOptions={
        'noData': -9999
    }
    # ^ Ensures correct NoData values
)

slope_task.start()
print("Slope export task initiated:", FILE_PREFIX_SLOPE)

# =============================================================================
# FINAL STATUS BLOCK
# =============================================================================

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("EXPORT PIPELINE INITIATED")
print("")
print("OPERATION ORDER:")
print("  1.  Load tiles (named band selection)")
print("  2.  Resample (bilinear) + reproject each tile (per-tile, PRE-mosaic)")
print("  3.  Mosaic reprojected tiles (newest tile wins)")
print("  4.  Assert analysis grid on mosaic output")
print("  5.  Export DEM (reproject → float → unmask → task)")
print("  6.  Compute slope on locked-grid DEM")
print("  7.  Assert analysis grid on slope output")
print("  8.  Convert degrees to percent slope")
print("  9.  Clip slope to AOI")
print(" 10.  Assert analysis grid post-clip (final spatial lock)")
print(" 11.  Cast to float32 + unmask NoData (non-spatial, applied last)")
print(" 12.  Export slope (explicit CRS + scale at task level)")
print("")
print("Grid Standard:     ", Export_CRS, "@", EXPORT_SCALE, "m")
print("Resample Method:   ", RESAMPLE_METHOD)
print("Mosaic Priority:    Newest tile wins (sort descending)")
print("Derivative Method:  ee.Terrain.slope")
print("Output — Slope:     Percent slope, float32")
print("Output — DEM:       Elevation (meters), float32")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")